# LangSmith Trace 过滤与 Agent 输出关系提取

目标：
- 保留每次整体 workflow 的基础信息
- 保留每个 agent 的输出
- 保留 agent 输出之间的关系（父子链路 + goto 跳转）

输入文件：`project_traces_task_filter.json`
输出文件：`project_traces_task_filter.filtered.json`

In [6]:
import json
from pathlib import Path
from datetime import datetime
import os

# set current working directory
# os.chdir('src/workplace/memorysrc')

INPUT_PATH = Path('project_traces_task_filter.json')
OUTPUT_PATH = Path('project_traces_task_filter.filtered.json')

# 指定要保留的 agent 名称；若设为 None 则自动识别（chain 且有 outputs）
AGENT_NAMES = {'manager', 'executor', 'tool_developer', 'integrator', 'agent', 'enhance_tools', 'context_summary'}

# 框架节点，默认不视作 agent
EXCLUDE_NAMES = {'LangGraph', 'RunnableSequence'}

raw = json.loads(INPUT_PATH.read_text())
print(f'Loaded traces: {len(raw)}')

Loaded traces: 15


In [7]:
def parse_time(ts):
    if not ts:
        return None
    try:
        if ts.endswith('Z'):
            ts = ts[:-1] + '+00:00'
        return datetime.fromisoformat(ts)
    except Exception:
        return None

def duration_ms(start_time, end_time):
    s = parse_time(start_time)
    e = parse_time(end_time)
    if not s or not e:
        return None
    return int((e - s).total_seconds() * 1000)

def compact_value(v, max_len=1500):
    if v is None:
        return None
    if isinstance(v, (str, int, float, bool)):
        s = str(v)
        return s[:max_len] if len(s) > max_len else v
    if isinstance(v, list):
        out = []
        for item in v[:8]:
            out.append(compact_value(item, max_len=max_len))
        return out
    if isinstance(v, dict):
        out = {}
        for k, val in v.items():
            if k in {'response_metadata', 'additional_kwargs', 'usage_metadata', 'llm_output', 'run'}:
                continue
            out[k] = compact_value(val, max_len=max_len)
        return out
    return str(v)[:max_len]

def extract_message_text(messages):
    texts = []
    if not isinstance(messages, list):
        return texts

    for msg in messages:
        content = msg.get('content') if isinstance(msg, dict) else None
        if isinstance(content, str):
            if content.strip():
                texts.append(content.strip())
        elif isinstance(content, list):
            parts = []
            for block in content:
                if isinstance(block, dict) and block.get('type') == 'text':
                    t = (block.get('text') or '').strip()
                    if t:
                        parts.append(t)
            if parts:
                texts.append('\n'.join(parts))
    return texts

def extract_tool_calls(messages):
    calls = []
    if not isinstance(messages, list):
        return calls

    for msg in messages:
        if not isinstance(msg, dict):
            continue
        tool_calls = msg.get('tool_calls')
        if not isinstance(tool_calls, list):
            continue
        for c in tool_calls:
            if not isinstance(c, dict):
                continue
            calls.append({
                'id': c.get('id'),
                'name': c.get('name'),
                'type': c.get('type'),
                'args': compact_value(c.get('args')),
            })
    return calls

def message_key(msg, idx):
    if not isinstance(msg, dict):
        return f'non_dict::{idx}::{str(msg)[:120]}'

    msg_id = msg.get('id')
    if msg_id:
        return f'id::{msg_id}'

    # fallback：部分消息无 id 时，用稳定指纹避免重复保存
    payload = {
        'type': msg.get('type'),
        'content': msg.get('content'),
        'name': msg.get('name'),
        'tool_call_id': msg.get('tool_call_id'),
        'tool_calls': msg.get('tool_calls'),
    }
    return f'fingerprint::{json.dumps(payload, ensure_ascii=False, sort_keys=True)}'

def keep_only_new_messages(messages, seen_keys):
    if not isinstance(messages, list):
        return []

    new_msgs = []
    for i, msg in enumerate(messages):
        key = message_key(msg, i)
        if key in seen_keys:
            continue
        seen_keys.add(key)
        new_msgs.append(msg)
    return new_msgs

def should_keep_as_agent(node, root_name):
    if not isinstance(node, dict):
        return False
    name = node.get('name')
    if not name or name == root_name or name in EXCLUDE_NAMES:
        return False

    outputs = node.get('outputs')
    if not isinstance(outputs, dict) or not outputs:
        return False

    if AGENT_NAMES is not None:
        return name in AGENT_NAMES

    return node.get('run_type') == 'chain'

def extract_agent_input(node):
    inputs = node.get('inputs') or {}
    res = {}

    if 'user_query' in inputs:
        res['user_query'] = compact_value(inputs.get('user_query'))

    if 'messages' in inputs:
        prompt_texts = extract_message_text(inputs.get('messages'))
        if prompt_texts:
            res['prompt_texts'] = prompt_texts

    return res

def extract_agent_output(node, seen_output_message_keys):
    outputs = node.get('outputs') or {}
    res = {}

    if 'final_answer' in outputs:
        res['final_answer'] = compact_value(outputs.get('final_answer'))

    if 'output' in outputs:
        res['output'] = compact_value(outputs.get('output'))

    if 'messages' in outputs:
        new_messages = keep_only_new_messages(outputs.get('messages'), seen_output_message_keys)
        msg_texts = extract_message_text(new_messages)
        if msg_texts:
            res['message_texts'] = msg_texts

        tool_calls = extract_tool_calls(new_messages)
        if tool_calls:
            res['tool_calls'] = tool_calls

    for key in ('tool_steps', 'retry_count', 'cumulative_tool_call_cnt'):
        if key in outputs:
            res[key] = outputs[key]

    return res

def extract_goto_targets(agent_output):
    targets = []
    out = agent_output.get('output')
    if isinstance(out, dict):
        goto = out.get('goto')
        if isinstance(goto, str):
            targets.append(goto)
        elif isinstance(goto, list):
            targets.extend([g for g in goto if isinstance(g, str)])
    return targets

def flatten_tree(root):
    all_nodes = []

    def walk(node):
        all_nodes.append(node)
        for ch in node.get('children') or []:
            walk(ch)

    walk(root)
    return all_nodes



In [8]:
filtered = []

for root in raw:
    root_name = root.get('name')
    root_outputs = root.get('outputs') if isinstance(root.get('outputs'), dict) else {}

    workflow = {
        'trace_id': root.get('trace_id'),
        'workflow_run_id': root.get('id'),
        'workflow_name': root_name,
        'start_time': root.get('start_time'),
        'end_time': root.get('end_time'),
        'duration_ms': duration_ms(root.get('start_time'), root.get('end_time')),
        'user_query': (root.get('inputs') or {}).get('user_query'),
        'workflow_final_answer': root_outputs.get('final_answer') if isinstance(root_outputs, dict) else None,
    }

    nodes = flatten_tree(root)
    id_to_node = {n.get('id'): n for n in nodes if n.get('id')}

    agent_runs = []
    agent_by_id = {}

    # 关键：按时间处理，才能正确做“只保留新输出”的差量提取
    nodes_sorted = sorted(nodes, key=lambda n: (n.get('start_time') or '', n.get('id') or ''))
    seen_output_message_keys = set()

    for n in nodes_sorted:
        if not should_keep_as_agent(n, root_name):
            continue

        agent_input = extract_agent_input(n)
        agent_output = extract_agent_output(n, seen_output_message_keys)
        rec = {
            'run_id': n.get('id'),
            'parent_run_id': n.get('parent_run_id'),
            'agent_name': n.get('name'),
            'run_type': n.get('run_type'),
            'start_time': n.get('start_time'),
            'end_time': n.get('end_time'),
            'duration_ms': duration_ms(n.get('start_time'), n.get('end_time')),
            'input': agent_input,
            'output': agent_output,
        }
        agent_runs.append(rec)
        agent_by_id[rec['run_id']] = rec

    # 关系边 1：最近上游 agent -> 当前 agent（沿 parent_run_id 向上回溯）
    edges = []
    seen = set()

    for run in agent_runs:
        cur_parent = run.get('parent_run_id')
        while cur_parent:
            if cur_parent in agent_by_id:
                key = (cur_parent, run['run_id'], 'parent_agent')
                if key not in seen:
                    seen.add(key)
                    edges.append({
                        'from_run_id': cur_parent,
                        'from_agent': agent_by_id[cur_parent]['agent_name'],
                        'to_run_id': run['run_id'],
                        'to_agent': run['agent_name'],
                        'relation': 'parent_agent',
                    })
                break
            parent_node = id_to_node.get(cur_parent)
            cur_parent = parent_node.get('parent_run_id') if isinstance(parent_node, dict) else None

    # 关系边 2：基于 output.goto 的跳转（符号关系）
    for run in agent_runs:
        targets = extract_goto_targets(run.get('output') or {})
        for tgt in targets:
            key = (run['run_id'], tgt, 'goto')
            if key in seen:
                continue
            seen.add(key)
            edges.append({
                'from_run_id': run['run_id'],
                'from_agent': run['agent_name'],
                'to_agent': tgt,
                'relation': 'goto',
            })

    filtered.append({
        'workflow': workflow,
        'agent_outputs': agent_runs,
        'relations': edges,
    })

OUTPUT_PATH.write_text(json.dumps(filtered, ensure_ascii=False, indent=2))
print(f'Wrote: {OUTPUT_PATH}')
print(f'Workflows: {len(filtered)}')
print(f'Agent runs (total): {sum(len(x["agent_outputs"]) for x in filtered)}')
print(f'Relations (total): {sum(len(x["relations"]) for x in filtered)}')



Wrote: project_traces_task_filter.filtered.json
Workflows: 15
Agent runs (total): 470
Relations (total): 418


In [9]:
# 查看一个样本
sample = filtered[0]

# save sample to json
with open('sample.json', 'w') as f:
    json.dump(sample, f, ensure_ascii=False, indent=2)
